In [ ]:
from dotenv import load_dotenv
import os
from langchain_ollama import OllamaEmbeddings
from langchain_postgres.vectorstores import PGVector
from langchain_core.documents import Document

load_dotenv(override=True)

PGVECTOR_ID = os.getenv("PGVECTOR_ID")
PGVECTOR_PW = os.getenv("PGVECTOR_PW")
PGVECTOR_HOST = os.getenv("PGVECTOR_HOST", "localhost")
PGVECTOR_PORT = os.getenv("PGVECTOR_PORT", "5432")
PGVECTOR_DB = os.getenv("PGVECTOR_DB")

connection = f'postgresql+psycopg://{PGVECTOR_ID}:{PGVECTOR_PW}@{PGVECTOR_HOST}:{PGVECTOR_PORT}/{PGVECTOR_DB}'

In [3]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
documents = loader.load()

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splitted_documents = text_splitter.split_documents(documents)

In [5]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="bge-m3:latest",
)

In [6]:
db = PGVector.from_documents(
    splitted_documents,
    embedding_model,
    connection=connection
)

In [9]:
query = "결혼하면 얼마를 받을 수 있어?"

In [10]:
docs = db.similarity_search(query, k=3)

In [11]:
docs

[Document(id='abb9488d-8cf5-4fd1-af8d-c7a784045902', metadata={'page': 11, 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx', 'Author': '', 'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'total_pages': 22, 'CreationDate': "D:20260116002528+09'00'"}, page_content='구분 대상 휴가 (일수) 경조금 (만원) 화환/조화\n결혼 본인 5 일 100 + 화환 지원\n자녀 1 일 50 + 화환 지원\n30 -\n형제/자매 1 일\n30 -\n회갑/칠순 본인/배우자 부모 1 일\n출산 본인 출산휴가 (90 일) 출산 축하금 50 과일 바구니\n배우자 10 일 (유급) 출산 축하금 50 과일 바구니\n-\n사망 본인/배우자 500 + 장례용품 3 단 조화 + 근조기\n부모/배우자 부모 5 일 100 + 장례용품 3 단 조화 + 근조기\n30\n조부모/외조부모 3 일 조화\n30\n형제/자매 3 일 조화'),
 Document(id='5d45d846-582c-4398-b479-97c58baf7d2a', metadata={'page': 12, 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx', 'Author': '', 'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft

In [12]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    '''다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트:{context}

질문: {question}
'''
)

prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 컨텍스트만 사용해 질문에 답하세요.\n컨텍스트:{context}\n\n질문: {question}\n')

In [16]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.output_parsers import StrOutputParser

# llm = OllamaLLM(model="gemma3:1b")
llm = OllamaLLM(model="gemma4:e2b")

chain = prompt | llm | StrOutputParser()

In [17]:
response = chain.invoke({'context': docs, 'question': query})

In [15]:
response

'결혼 본인 5 일 100 + 화환 지원\n'